In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))  # add repo root
from rsr import rsr
import torch
import json
import pandas as pd

from ndtools import fun_binary_graph as fbg # ndtools available at github.com/jieunbyun/network-datasets
from ndtools.graphs import build_graph
from pathlib import Path
import networkx as nx   

# Load data

In [2]:
DATASET = Path("data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs_bin.json").read_text(encoding="utf-8"))

# build base graph
G_base: nx.Graph = build_graph(nodes, edges, probs_dict)


In [3]:
row_names = list(edges.keys())
n_state = 2 # binary states

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

In [4]:
#origin = 'n1'
origin = 'n52'
dests = ['n22', 'n66']
sys_upper_st = 1

def s_fun(comps_st):
    travel_time, sys_st, info = fbg.eval_travel_time_to_nearest(comps_st, G_base, origin, dests,
                                                         avg_speed=60, # km/h
                                                         target_max = [1.5, 0.5], # hours: it shouldn't take longer than this compared to the original travel time
                                                         length_attr = 'length_km')
    if sys_st >= sys_upper_st:
       path = info['path_filtered_edges'] 
       min_comps_st = {eid: ('>=', 1) for eid in path} # edges in the path are working
       min_comps_st['sys'] = ('>=', sys_st) # system edge is also working
    else:
        min_comps_st = None
    return travel_time, sys_st, min_comps_st

In [7]:
RSRPATH = Path("rsr_res") 

refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
refs_mat_upper = refs_mat_upper.to(device)
refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
refs_mat_lower = refs_mat_lower.to(device)

# Calculate system probabilities

## Marginal probability

In [10]:
pr_cond = rsr.get_comp_cond_sys_prob(
    refs_mat_upper,
    refs_mat_lower,
    probs,
    comps_st_cond = {},
    row_names = row_names,
    s_fun = s_fun,
    sys_upper_st = sys_upper_st
)
print(f"P(sys >= {sys_upper_st}) = {pr_cond['upper']:.3e}")
print(f"P(sys <= {sys_upper_st-1} ) = {pr_cond['lower']:.3e}\n")

P(sys >= 1) = 9.987e-01
P(sys <= 0 ) = 1.319e-03



## Conditional probability given "one" component's survival

In [11]:
for x in row_names:
    print(f"Eval P(sys | {x}=1)")
    pr_cond = rsr.get_comp_cond_sys_prob(
        refs_mat_upper,
        refs_mat_lower,
        probs,
        comps_st_cond = {x: 1},
        row_names = row_names,
        s_fun = s_fun,
        sys_upper_st = sys_upper_st
    )
    print(f"P(sys >= {sys_upper_st} | {x}=1) = {pr_cond['upper']:.3e}")
    print(f"P(sys <= {sys_upper_st-1} | {x}=1) = {pr_cond['lower']:.3e}\n")

Eval P(sys | e0001=1)
P(sys >= 1 | e0001=1) = 9.986e-01
P(sys <= 0 | e0001=1) = 1.395e-03

Eval P(sys | e0002=1)
P(sys >= 1 | e0002=1) = 9.986e-01
P(sys <= 0 | e0002=1) = 1.417e-03

Eval P(sys | e0003=1)
P(sys >= 1 | e0003=1) = 9.987e-01
P(sys <= 0 | e0003=1) = 1.320e-03

Eval P(sys | e0004=1)
P(sys >= 1 | e0004=1) = 9.987e-01
P(sys <= 0 | e0004=1) = 1.339e-03

Eval P(sys | e0005=1)
P(sys >= 1 | e0005=1) = 9.986e-01
P(sys <= 0 | e0005=1) = 1.367e-03

Eval P(sys | e0006=1)
P(sys >= 1 | e0006=1) = 9.986e-01
P(sys <= 0 | e0006=1) = 1.371e-03

Eval P(sys | e0007=1)
P(sys >= 1 | e0007=1) = 9.986e-01
P(sys <= 0 | e0007=1) = 1.381e-03

Eval P(sys | e0008=1)
P(sys >= 1 | e0008=1) = 9.986e-01
P(sys <= 0 | e0008=1) = 1.364e-03

Eval P(sys | e0009=1)
P(sys >= 1 | e0009=1) = 9.987e-01
P(sys <= 0 | e0009=1) = 1.322e-03

Eval P(sys | e0010=1)
P(sys >= 1 | e0010=1) = 9.986e-01
P(sys <= 0 | e0010=1) = 1.368e-03

Eval P(sys | e0011=1)
P(sys >= 1 | e0011=1) = 9.987e-01
P(sys <= 0 | e0011=1) = 1.341e-03


# Get multi-state probabilities

## First load refs for all states

In [13]:
RSRPATH = Path("rsr_res") 
refs_dict_mat_upper = {}
refs_dict_mat_lower = {}

for sys_upper_st in [1, 2]:  # either 1 or 2
    refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
    refs_mat_upper = refs_mat_upper.to(device)
    refs_dict_mat_upper[sys_upper_st] = refs_mat_upper
    refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
    refs_mat_lower = refs_mat_lower.to(device)
    refs_dict_mat_lower[sys_upper_st] = refs_mat_lower

## Marginal probability

In [14]:

# Initialize empty list to store the results
results = []

# Calculate probabilities
cond_probs = rsr.get_comp_cond_sys_prob_multi(
                refs_dict_mat_upper,
                refs_dict_mat_lower,
                probs,
                comps_st_cond = {},
                row_names = row_names,
                s_fun = s_fun
            )

# Print results
print("P(sys):", cond_probs)

# Append data as a dictionary to the list
results.append({"System failure": cond_probs[0],
                "Partial failure": cond_probs[1],
                "Survival": cond_probs[2]
                })

# Convert the list to a DataFrame
df_results = pd.DataFrame(results)

# Save to a JSON file
df_results.to_json("post-processing/sys_probs.json", orient="records", indent=4)

print("\nData saved to post-processing/sys_probs.json")

P(sys): {0: 0.001364, 1: 0.009518, 2: 0.989118}

Data saved to post-processing/sys_probs.json


## Conditional probability given "one" component's survival
State 0: System failure\
State 1: Partial failure\
State 2: System survival

In [15]:
# Initialize empty list to store the results
results = []

for x in row_names:
    # Calculate probabilities
    cond_probs = rsr.get_comp_cond_sys_prob_multi(
                    refs_dict_mat_upper,
                    refs_dict_mat_lower,
                    probs,
                    comps_st_cond = {x: 0}, # 1: survival, 0: failure
                    row_names = row_names,
                    s_fun = s_fun
                )

    # Print results
    print(f"P(sys | {x}=0):", cond_probs)

    # Append data as a dictionary to the list
    results.append({"Component": x,
                    "System failure": cond_probs[0],
                    "Partial failure": cond_probs[1],
                    "Survival": cond_probs[2]
                    })

# Convert the list to a DataFrame
df_results = pd.DataFrame(results)

# Save to a JSON file
df_results.to_json("post-processing/cond_sys_probs.json", orient="records", indent=4)

print("\nData saved to post-processing/cond_sys_probs.json")

P(sys | e0001=0): {0: 0.001416, 1: 0.009528, 2: 0.989056}
P(sys | e0002=0): {0: 0.001359, 1: 0.00962, 2: 0.989021}
P(sys | e0003=0): {0: 0.001369, 1: 0.009607, 2: 0.989024}
P(sys | e0004=0): {0: 0.001342, 1: 0.009534, 2: 0.989124}
P(sys | e0005=0): {0: 0.001346, 1: 0.009637, 2: 0.989017}
P(sys | e0006=0): {0: 0.001369, 1: 0.009585, 2: 0.989046}
P(sys | e0007=0): {0: 0.001399, 1: 0.009605, 2: 0.988996}
P(sys | e0008=0): {0: 0.001284, 1: 0.009652, 2: 0.989064}
P(sys | e0009=0): {0: 0.001329, 1: 0.00943, 2: 0.989241}
P(sys | e0010=0): {0: 0.00135, 1: 0.009471, 2: 0.989179}
P(sys | e0011=0): {0: 0.001348, 1: 0.009567, 2: 0.989085}
P(sys | e0012=0): {0: 0.001333, 1: 0.009451, 2: 0.989216}
P(sys | e0013=0): {0: 0.00139, 1: 0.00933, 2: 0.98928}
P(sys | e0014=0): {0: 0.00131, 1: 0.009659, 2: 0.989031}
P(sys | e0015=0): {0: 0.001332, 1: 0.009523, 2: 0.989145}
P(sys | e0016=0): {0: 0.001358, 1: 0.009454, 2: 0.989188}
P(sys | e0017=0): {0: 0.001394, 1: 0.009622, 2: 0.988984}
P(sys | e0018=0): {0: